<a href="https://colab.research.google.com/github/Prerna-Karle/genai-workshop/blob/main/AI_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers ipywidgets wikipedia

import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import wikipedia
import re
from transformers import pipeline

print("Booting up the Agent Controller (Downloading lightweight NLP model)...")
# We use a zero-shot classifier as the "brain" to perfectly route the agent's decisions
agent_brain = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("Agent Ready!")

# ==========================================
# 1. DEFINE THE TOOLS (The Agent's "Hands")
# ==========================================
def tool_wikipedia_search(query):
    """Searches Wikipedia and returns the live summary."""
    try:
        # Get the most relevant page title
        search_results = wikipedia.search(query)
        if not search_results:
            return "Observation: No Wikipedia page found for this topic."
        # Fetch the summary of the top result
        summary = wikipedia.summary(search_results[0], sentences=2)
        return f"Observation: According to Wikipedia, {summary}"
    except Exception as e:
        return f"Observation: Wikipedia search failed due to ambiguity - try being more specific."

def tool_financial_calculator(expression):
    """Safely evaluates basic math for ROI and finances."""
    try:
        # Extract only math symbols and numbers for safety
        clean_expr = re.sub(r'[^0-9\+\-\*\/\(\)\.]', '', expression)
        result = eval(clean_expr) # Executes real python math
        return f"Observation: The calculated financial result is {result}"
    except:
        return "Observation: Calculation failed. Invalid numbers provided."

# ==========================================
# 2. THE AGENTIC LOOP (The ReAct Architecture)
# ==========================================
def run_autonomous_agent(user_prompt):
    print(f"\n[USER PROMPT]: {user_prompt}\n" + "="*55)

    # --- STEP 1: THINKING & PLANNING ---
    print("🧠 [THOUGHT]: Analyzing intent to select the best tool...")
    time.sleep(0.5)

    # The LLM looks at the prompt and scores it against our available tools
    tools_available = [
        "Search Wikipedia for factual knowledge",
        "Calculate numbers and financial math",
        "General conversation"
    ]
    decision = agent_brain(user_prompt, candidate_labels=tools_available)
    chosen_tool = decision['labels'][0]

    # --- STEP 2: ACTION (TOOL CALLING) ---
    observation = ""
    if chosen_tool == "Search Wikipedia for factual knowledge":
        # Strip out conversational words to feed a clean query to the API
        subject = re.sub(r'(?i)(who is|what is|tell me about|search for)', '', user_prompt).strip()
        print(f"🛠️ [ACTION]: Calling Tool -> `tool_wikipedia_search(query='{subject}')`")
        observation = tool_wikipedia_search(subject) # Actually hitting the web!

    elif chosen_tool == "Calculate numbers and financial math":
        print(f"🛠️ [ACTION]: Calling Tool -> `tool_financial_calculator(expression='{user_prompt}')`")
        observation = tool_financial_calculator(user_prompt) # Actually running math!

    else:
        print("🛠️ [ACTION]: No external tool needed.")
        observation = "Observation: This is a general query."

    # --- STEP 3: OBSERVATION ---
    print(f"👀 [OBSERVATION]: {observation}")
    time.sleep(0.5)

    # --- STEP 4: FINAL ANSWER ---
    print("\n✅ [FINAL ANSWER]:")
    if "Wikipedia" in observation:
        print(f"Based on my live web search: {observation.replace('Observation: According to Wikipedia, ', '')}")
    elif "calculated" in observation:
        print(f"I ran the numbers for you: {observation.replace('Observation: ', '')}")
    else:
        print("I don't need a tool for this! Ask me a math question or to look up a fact.")
    print("="*55)

# ==========================================
# 3. INTERACTIVE UI FOR COLAB
# ==========================================
agent_input = widgets.Text(
    value="find wikipedia information on Google",
    description='Ask Agent:',
    layout=widgets.Layout(width='80%')
)
agent_btn = widgets.Button(description="Run AI Agent", button_style='warning')
agent_output = widgets.Output()

def on_click_agent(b):
    with agent_output:
        clear_output()
        run_autonomous_agent(agent_input.value)

agent_btn.on_click(on_click_agent)
display(agent_input, agent_btn, agent_output)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 26.3 MB/s eta 0:00:00
Booting up the Agent Controller (Downloading lightweight NLP model)...


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Agent Ready!


Text(value='find wikipedia information on Google', description='Ask Agent:', layout=Layout(width='80%'))

Button(button_style='warning', description='Run AI Agent', style=ButtonStyle())

Output()